In [ ]:
from collections.abc import Callable, Iterable
import json
from pathlib import Path

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData

from jazz_graph.data.graph_builder.graph_builder import CreateTensors
from jazz_graph.data.fetch import fetch_recording_traits, fetch_discogs_to_recording_id
from jazz_graph.model.model import LinkPredictionModel, UnsupervisedJazzModel
from jazz_graph.recommendation.recommender import Recommender, LookupRecordings, PredictLinkRecommender, RandomWalkRecommender
from jazz_graph.metrics.ranking import map_at_k
from jazz_graph.recommendation.experiment import BSideExperiment
from jazz_graph.training.logging import ExperimentLogger, load_embeddings, load_model, find_most_recent_run

checkpoint_path = None
# checkpoint_path = '/workspace/experiments/2026-03-26_02-01-03_gnn_simCLR_graph_parquet'  # This was good, but I think we need to retrain on new data.
# checkpoint_path = '/workspace/experiments/2026-04-03_14-48-08_gnn_simCLR_graph_parquet'  # this is a simple edge data model, which fails the b-side test.
checkpoint_path = '/workspace/experiments/2026-04-03_16-48-18_gnn_simCLR_graph_parquet'
if checkpoint_path is None:
    checkpoint_path = str(find_most_recent_run('/workspace/experiments', 'simCLR'))

with open(Path(checkpoint_path) / 'config.json', 'r') as f:
    run_config = json.loads(f.read())
    nodes_data_path = run_config['data_config'].get('dataset')

experiment_config = {
    'gnn': checkpoint_path,
    'nodes_data': nodes_data_path,
    'recommender_pooling': 'softmax'
}


In [ ]:
def get_coverage(rec_ids, recording_traits: pd.DataFrame):
    recommended: pd.DataFrame = recording_traits.loc[rec_ids]
    artists = recommended['artist']
    artist_counts = artists.value_counts(normalize=True)
    return artist_counts.to_dict(), len(artist_counts) / len(rec_ids)


def get_run_name(experiment_config: dict) -> str:
    run_config = experiment_config['run_configuration']
    task = run_config.get('training_task')
    if task is None:
        return "official_match_album_no_features"
    return 'official_' + task +'_with_features'

recording_traits = fetch_recording_traits(use_proto=nodes_data_path.endswith('proto')).set_index('recording_id')


In [ ]:
import os
import jsonlines

def get_config_details(path):
    ex_config = path / 'config.json'
    with open(ex_config) as f:
        conf = json.loads(f.read())
    try:
        run_name = get_run_name(conf)
    except Exception:
        run_name = conf['model']
    pooling = conf.get('recommender_pooling', '---')
    model_name = run_name[len('official_'):] if run_name.startswith('official') else run_name
    return model_name, pooling

def get_spotify_results(path: Path):
    model_name, pooling = get_config_details(path)
    metrics_path = path / 'metrics.jsonl'
    metric = next(iter(jsonlines.open(metrics_path)))
    coverage = metric['unseeded_coverage']['novel_frequency']
    return [metric['recall_novel'], metric['recall_familiar'], coverage]

def get_bside_results(path: Path):
    metrics_path = path / 'metrics.jsonl'
    n_experiments = 0
    map_at_k = 0
    for bside in jsonlines.open(metrics_path):
        n_experiments += 1
        map_at_k += bside['MAP_at_k']
    return map_at_k / n_experiments

def traverse_official():
    """Report on the official experiment scores."""
    dir = Path('../experiments')
    results = {}
    for path in dir.iterdir():
        if 'backup' in path.name:
            continue
        if not 'official' in path.name:
            continue
        for e in path.iterdir():
            model_name, pooling = get_config_details(e)
            if (model_name, pooling) not in results:
                results[(model_name, pooling)] = {}
            if 'bside' in e.name:
                bside = get_bside_results(e)
                results[(model_name, pooling)]['bside'] = bside
            else:
                 results[(model_name, pooling)]['spotify'] = get_spotify_results(e)
    row = []
    for key, value in results.items():
        spotify = value['spotify']
        bside = [value['bside']]
        row.append([key[0], key[1], *(spotify + bside)])
    return pd.DataFrame.from_records(row, columns=['model', 'pooling', 'novel_recall', 'familiar_recall', 'coverage', 'BSide_mean_MAP_at_k'])


In [ ]:
traverse_official().sort_values('novel_recall')

In [ ]:
# for path in find_most_recent_run('../experiments', 'simCLR', return_all=True):
#     with open(path / 'config.json') as f:
#         config = json.loads(f.read())
#         if config.get('training_task') is None:
#             continue
#         print(path, config)


In [ ]:
from jazz_graph.training.logging import clean_up_experiments

clean_up_experiments('../experiments', 'simCL', dry_run=True)

In [ ]:
run_config

In [ ]:
def lookup_from_dataset(data: HeteroData) -> LookupRecordings:
    # But why? Answer: the graph data may include pruning that is not part of the
    # parquet datasets or recording traits.
    rec_ids = data['performance'].x[:, 1].numpy()
    ids = np.arange(rec_ids.size)
    df = pd.DataFrame({'recording_ids': rec_ids, 'ids': ids}).set_index('recording_ids')
    return LookupRecordings(df)


In [ ]:
class UnsupervisedModelAdapter(torch.nn.Module):
    def __init__(self, model: UnsupervisedJazzModel):
        super().__init__()
        self.model = model

    def __call__(self, x_dict, edge_index_dict, batch):
        return self.model(batch)
        return self.model.encode(batch)

In [ ]:
def make_jazz_graph_from_config(config):
    graph_data_function = config.get('graph_data_function')
    nodes_data_path = config['data_config']['dataset']
    if graph_data_function is None:
        from jazz_graph.data.graph_builder.graph_builder import make_jazz_data, CreateTensors
        return  make_jazz_data(CreateTensors(nodes_data_path))
    from jazz_graph.data.graph_builder import make_jazz
    make_jazz_graph = getattr(make_jazz, graph_data_function)
    store = make_jazz.JazzDataStore(nodes_data_path)
    return make_jazz_graph(store)

In [ ]:
## Inductive Graph Recommender
from sklearn import model_selection

from jazz_graph.model.model import JazzModel
from jazz_graph.recommendation.recommender import InferenceRecommender


graph_data: HeteroData = make_jazz_graph_from_config(run_config)
model_state = load_model(checkpoint_path)
model_state = model_state.get('model_state_dict', model_state)
model_state = model_state.get('model_state_dict', model_state)
model = UnsupervisedJazzModel.from_config(run_config)
model.load_state_dict(model_state)
model = UnsupervisedModelAdapter(model)
recommender = InferenceRecommender(model, graph_data, pooling=experiment_config['recommender_pooling'])
lookup = recommender.lookup_recordings
recording_traits = fetch_recording_traits(use_proto=nodes_data_path.endswith('proto')).set_index('recording_id')
experimenter = BSideExperiment(recommender, recording_traits, '../experiments')

In [ ]:
graph_data

In [ ]:

album_experiments = [
    ('Miles Davis', 'Kind of Blue'),
    ('Miles Davis', 'Sketches of Spain'),
    ('Art Blakey & The Jazz Messengers', 'Mosaic'),
    ('Charles Mingus', "Mingus Ah Um"),  # lots of songs, should have some.
    ('The Dave Brubeck Quartet', "Time Out"),
    ('Ornette Coleman', 'The Shape of Jazz to Come')  # very unusual music--should probably be easy.
]

In [ ]:
# experiment = album_experiments[0]
# print(experiment)
# result = experimenter.b_side_precision(*experiment)
# result
# recording_traits.loc[result['recommended_ids']]


In [ ]:
# results = experimenter.b_side_experiment(experiment_config, album_experiments)
# pd.DataFrame.from_records(results).drop(columns=['recommended_ids'])

## Spotify Data

In [ ]:
import os
import json
from jazz_graph.recommendation.playlist import SpotifyListens
from jazz_graph.recommendation.recommender import ArtistWeightedRecommender
from jazz_graph.clean.data_normalization import normalize_title
import random

from jazz_graph.recommendation.experiment import SpotifyExperiement, RandomAlbumSplit


In [ ]:
spotify = 'local_data'
spotify_sample = '../local_data/my_spotify_data/spotify_extended_streaming_history/Streaming_History_Audio_2024-2025_4.json'
with open(spotify_sample, 'r') as f:
    spotify_data = json.loads(f.read())

def spotify_details(record: dict):
    details = [
        'ts', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name',
        'reason_start', 'reason_end', 'shuffle', 'skipped'
    ]
    return {key: record.get(key) for key in details}

spotify_data[-100:-90]
spotify_data[-10:]
ten_recent = [spotify_details(record) for record in spotify_data[-20:-10]]

In [ ]:

# seen = set()
# i = 0
# for rec in spotify_data:
#     title = rec['master_metadata_track_name']
#     norm_title = normalize_title(title)
#     if norm_title in seen:
#         continue
#     seen.add(norm_title)
#     if 'digital' in title.lower():
#         i += 1
#         print(norm_title)
#         print(title)
#     # print(norm_title)
# print(i)

In [ ]:
def test_rand_walk_recommender(graph_data):
    kob = recording_traits[recording_traits.album == "Kind of Blue"]
    baseline_recommender = RandomWalkRecommender(graph_data, 42)
    recommendations, scores, mask = baseline_recommender.get_recommendations(kob.index.to_list())
    ex_shape = recommendations.shape
    assert scores.shape == ex_shape
    assert mask.dtype == bool
    assert mask.shape == ex_shape
    return recommendations


# test_recs = test_rand_walk_recommender(graph_data)
# recording_traits.loc[test_recs]

In [ ]:
baseline_recommender = RandomWalkRecommender(graph_data, 42)
simple_artist_recommender = ArtistWeightedRecommender(recording_traits)


In [ ]:
# fields = 'master_metadata_track_name', 'master_metadata_album_album_name', 'master_metadata_album_artist_name'
# for recording in spotify_data:
#     for field in fields:
#         if recording.get(field) is None:
#             print(recording)

In [ ]:
experiment = SpotifyExperiement(recording_traits, spotify_data, seed=42, log_dir='../experiments')

In [ ]:
# experiment.run_experiment(recommender, experiment_config, k=20)


In [ ]:
recommendations, scores, mask = recommender.get_recommendations(experiment.in_samples)

In [ ]:
recommendations.size

In [ ]:
all_out_samples = np.isin(recommendations, experiment.out_samples)
import matplotlib.pyplot as plt
plt.hist(scores[all_out_samples])

In [ ]:
misses = np.isin(recommendations[13_000:], experiment.out_samples)
misses = recommendations[13_000:][misses]
hits = np.isin(recommendations[:13_000], experiment.out_samples)
hits = recommendations[:13_000][hits]
len(hits), len(misses)

In [ ]:
recording_traits.loc[hits]

In [ ]:
lookup.lookup_node_index([6697991, 467832])

In [ ]:
graph_data[('artist', 'performs', 'performance')].edge_index
graph_data['performs'].edge_index
mask_perf_edge = torch.isin(graph_data['performs'].edge_index[1], torch.tensor([ 5207, 41607]))
graph_data['performs'].edge_index[:, mask_perf_edge]

In [ ]:
from torch_geometric.utils import subgraph
assert torch.any(torch.isin(graph_data['performs'].edge_index[1], torch.tensor([5207])))
subgraph(torch.tensor([5207]), graph_data['performs'].edge_index)  # I expceted this to be the edge index connecting node 5207! It's empty.

In [ ]:
recording_traits.loc[hits].head(20)

In [ ]:
recording_traits.loc[misses].head(20)

In [ ]:
def load_rec_run(path):
    path = Path(path)
    with open(path / 'metrics.jsonl') as f:
        return json.loads(f.read())

results = load_rec_run(str(find_most_recent_run('/workspace/experiments', 'spotify_recommendations')))
results['seeded_coverage']['novel_frequency'], results['unseeded_coverage']['novel_frequency']
results['seeded_coverage']
results['unseeded_coverage']

In [ ]:

samples = results['samples']
assert np.setdiff1d(samples['true_positive_in_familiar_recommendation'], samples['true_positive_in_familiar_recommendation']).size == 0

In [ ]:
tp_familiar = results['samples']['true_positive_in_familiar_recommendation']
recording_traits.loc[tp_familiar].value_counts('artist')

In [ ]:
from functools import cache

def kind_of_blue_embeds(recommender, graph_data):
    LookupRecordings.from_hetero_data(graph_data)
    kob = recording_traits.query("album == 'Kind of Blue'").sort_values('release_date')
    kob_ids = kob[:5].index.to_list()
    kob_ids = lookup.lookup_node_index(kob_ids)
    embeds = recommender.model.model(graph_data)['performance']
    kob_embeds = embeds[kob_ids, :]
    rand_embeds = torch.argsort(torch.rand(embeds.size(0)))
    return kob_embeds, embeds[rand_embeds[:5]]

kob, rand = kind_of_blue_embeds(recommender, graph_data)

In [ ]:
kob_embed, rand_embed = kob, rand
((kob_embed @ kob_embed.T) - torch.eye(5)).mean(), (kob_embed @ rand_embed.T).mean()


In [ ]:
false_negatives = results['samples']['false_negative_in_novel_sample']
true_positives = results['samples']['true_positive_in_novel_recommendation']
recording_traits.loc[false_negatives].sort_values('release_date')
recording_traits.loc[true_positives][:50]
novel_recs = results['samples']['top_1132_novel_recommendations']
recording_traits.loc[novel_recs].head(50)
new_novel_recs = np.setdiff1d(novel_recs, true_positives)
recording_traits.loc[novel_recs].value_counts('artist')[:10]


In [ ]:
n = 250
recording_traits.loc[new_novel_recs][n:n+50]  #.drop_duplicates('artist', keep='first')[:50]

In [ ]:
recording_traits.title.value_counts(normalize=True)

In [ ]:
# Autumn Leaves test.
# This song is one of the most recorded songs in jazz history.
# We anticipate that if it was the only song you had listened to, you would be recommended
# other performances of it.
def autmun_leaves(recommender: InferenceRecommender):
    # Two piano trios and a guitar quintet and a trobone quintet. The artist are disjoint (but very likely connected in few degrees.)
    # Still, show me Autumn Leaves more than random!
    seed_ids = [4920193, 7100264, 5950710, 6925518, 29157716]  # Vince Guaraldi, McCoy Tyner, Kenny Burrel w/ Cedar Walton, Benny Golson
    recs, scores, mask = recommender.get_recommendations(seed_ids)

    autumn_recs = recording_traits.loc[recs][:10000]
    seed_ids = [
        29157719,  # Someday my Prince will Come, Bill Evans
        6925522, # A Bit of Heaven, Benny Golson
        7100266, # When Sunny Gets Blue, McCoy Tyner
        5950714, # Lucky to Be Me, Kenny Burrell
        4920194, # Three Coins in the Fountain, Vince Guaraldi
    ] #
    recs, scores, mask = recommender.get_recommendations(seed_ids)
    alt_recs = recording_traits.loc[recs]
    return autumn_recs, alt_recs


autumn_recs, alt_recs = autmun_leaves(recommender)


In [ ]:
autumn_recs.assign(rank=np.arange(len(autumn_recs))).query('title == "Autumn Leaves" and rank < 10_000').shape

In [ ]:
alt_recs.assign(rank=np.arange(len(alt_recs))).query('title == "Autumn Leaves" and rank < 10_000').shape

In [ ]:
recording_traits.title.value_counts()[:50]

## Explore misses in Spotify Matching

In [ ]:
import re
recording_traits[(recording_traits.artist.str.contains('Coltrane')) & recording_traits.album.str.contains('live', flags=re.IGNORECASE)].head(50)
None

In [ ]:
find_misses = SpotifyListens(recording_traits)
misses = [e[0] for e in find_misses.get_spotify_misses(spotify_data)]
print(len(misses), len(spotify_data))

In [ ]:
[key for key in find_misses.lookup if key[2].startswith('wayne s') and 'ad' in key[1]]

In [ ]:
recording_traits[recording_traits['album'].str.contains("Adam")]

In [ ]:
misses[0]

In [ ]:
unique = set()
clean_misses = []
for record in misses:
    keys = 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name'

    details = spotify_details(record)
    key = tuple(details[e] for e in keys)
    if key in unique:
        continue
    details.pop('ts')

    e = {key: details[other_key] for key, other_key in zip(['song', 'artist', 'album', ], keys)}
    clean_misses.append(e)
    unique.add(key)


In [ ]:
clean_misses

In [ ]:
[e for e in clean_misses if 'Gloria' in e['song']]

In [ ]:


def match_recording_traits(recording_traits, artist=None, album=None):
    def starts_with_lower(key, match):
        value = recording_traits[key]
        return value.str.contains(match, flags=re.IGNORECASE)

    if artist and album:
        return recording_traits[starts_with_lower('artist', artist) & starts_with_lower('album', album)]
    elif artist:
        return recording_traits[starts_with_lower('artist', artist)]
    elif album:
        return recording_traits[starts_with_lower('album', album)]


In [ ]:
re.search('^(Sunday )?at the', 'At the Village', re.IGNORECASE)

In [ ]:

result = match_recording_traits(recording_traits, 'Wayne Shorter', album='Adam')
#print(len(result))
result.sort_values('release_date').sort_values('title')#.loc[1662631]

In [ ]:
sunday = [spotify_details(v) for v in misses if v['master_metadata_album_album_name'].startswith('Sunday')]
normalize_title(sunday[0]['master_metadata_track_name']) == normalize_title(result.sort_values('release_date').loc[1662631].title)


In [ ]:
(sunday[0]['master_metadata_track_name'], result.sort_values('release_date').loc[1662631].title)

In [ ]:

result = match_recording_traits(recording_traits, album="N")
print(len(result))
result

In [ ]:
recording_traits.loc[experiment.in_samples].query("artist.str.contains('John')")

In [ ]:
recording_traits.loc[experiment.out_samples].query("artist.str.contains('John')")

In [ ]:
novel_recs = recording_traits.loc[recommendations[~mask]][:500]
all_recs = recording_traits.loc[recommendations][:500].assign(rank=np.arange(1, 501))
novel_recs = novel_recs.merge(all_recs['rank'], left_index=True, right_index=True)
repeat_idx = all_recs.index.difference(novel_recs.index)
repeat_recs = all_recs.loc[repeat_idx]
repeat_recs.head(30)

In [ ]:
from jazz_graph.etl.extract_discogs import InMemDiscogs, is_jazz_album

discogs = InMemDiscogs('/workspace/local_data/jazz_releases.jsonl', is_jazz_album)

In [ ]:
mismatches =[
    ('A Love Supreme', "song titles can be weird w/ Pt. I etc."),
    ('The Bill Evans Trio', "THE"),
    ('Equinox', 'Album naima, JC. Missing in discogs.'),
    ("Ascenseur pour l'échafaud", "This was  sound track by Miles in 1957. It's borked.")
    ("Feio", "no discogs match (Bitches Brew track.)")
]

In [ ]:
discogs.tracklist().get(normalize_title('naima'))

In [ ]:
discogs.get_albums_matching_title(normalize_title('a love supreme'))

In [ ]:
import jsonlines
with jsonlines.open(discogs.release_path) as f:
    for release in f:
        if normalize_title(release['title']) == normalize_title('the art of conversation'):
            print(release)

In [ ]:
import psycopg
query = """
    SELECT * FROM recording_first_release
    WHERE
        album ILIKE 'ada%' AND
        artist ILIKE 'Wayne Shorter'
        -- AND title ILIKE 'eq%'
    LIMIT 100;
"""
with psycopg.connect("dbname=musicbrainz_db user=philosofool") as conn:
    query_result = pd.read_sql(query, conn)
query_result

In [ ]:
query_result.album.loc[0]

In [ ]:
for row in query_result.itertuples():
    print(discogs.matching_discog(row))

In [ ]:
from jazz_graph.etl.extract_discogs import MatchDiscogs, InMemDiscogs, is_jazz_album
discogs = MatchDiscogs(InMemDiscogs('/workspace/local_data/jazz_releases.jsonl', is_jazz_album))

In [ ]:
discogs.matching_discog((None, None, "Adam's Apple", "Adam's Apple", "Wayne Shorter"))

In [ ]:
embeddings = torch.tensor([
    [.1, .1],
    [1, 0],
    [.1, .2],
    [.2, .1],
    [.3, .3]
])

dp = embeddings @ embeddings[[1, 3]].T
dp
dp.sum(dim=-1)